# Car Brochure Hybrid Search
1. Parse all PDFs in a folder
2. Extract features per car via Claude
3. Write a single Markdown file (one section per car)
4. Hybrid search (ChromaDB + BM25) — no metadata

In [ ]:
!pip install pymupdf pdfplumber anthropic sentence-transformers chromadb rank_bm25 --quiet
print('✅ Done')

## Step 1 — Parse all PDFs in folder

In [ ]:
import pymupdf
import pdfplumber
import anthropic
import re
import json
from pathlib import Path

ANTHROPIC_API_KEY = "YOUR_ANTHROPIC_API_KEY"   # ← paste your key
PDF_FOLDER        = "."                         # ← folder containing your PDFs

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print('✅ Ready')

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Pull all text + tables from a PDF into one raw string.
    Uses PyMuPDF for text and pdfplumber for tables.
    """
    # ── Text via PyMuPDF ────────────────────────────────────────────────────
    doc = pymupdf.open(pdf_path)
    pages_text = []
    for page in doc:
        text = page.get_text('text').strip()
        if text:
            pages_text.append(text)
    doc.close()

    # ── Tables via pdfplumber ────────────────────────────────────────────────
    tables_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    clean = [cell.strip() if cell else '' for cell in row]
                    if any(clean):
                        tables_text.append(' | '.join(clean))

    combined = '\n'.join(pages_text)
    if tables_text:
        combined += '\n\n--- TABLES ---\n' + '\n'.join(tables_text)

    return re.sub(r'\n{3,}', '\n\n', combined).strip()


# Discover all PDFs in the folder
pdf_files = sorted(Path(PDF_FOLDER).glob('*.pdf'))
print(f'Found {len(pdf_files)} PDF(s):')
for p in pdf_files:
    print(f'  • {p.name}')

## Step 2 — Extract features per car via Claude

In [ ]:
EXTRACT_PROMPT = """
You receive raw text extracted from a car brochure PDF.
Extract and return a clean Markdown section for this car with the following structure:

## {Car Model Name}

### Overview
One or two sentences describing what this car is.

### Specifications
Key specs as a bullet list (engine, dimensions, transmission, brakes, etc.).
Only include specs that are actually in the text — do not guess.

### Features
All features mentioned (safety, comfort, technology, exterior, interior).
Each feature on its own bullet line with a short description if available.

### Variants
List the available variants/trims if mentioned.

### Colors
List color options if mentioned.

RULES:
- Output ONLY the Markdown section, nothing else.
- Do not add info that isn't in the source text.
- Keep original spec values and units exactly as written.
- Write in the same language as the brochure (Bahasa Indonesia is fine).
"""


def extract_features_with_claude(raw_text: str, filename: str) -> str:
    """Send raw PDF text to Claude, get back a clean Markdown section."""
    response = client.messages.create(
        model      = 'claude-sonnet-4-5',
        max_tokens = 2000,
        system     = EXTRACT_PROMPT,
        messages   = [{
            'role'   : 'user',
            'content': f'Source file: {filename}\n\nRaw brochure text:\n\n{raw_text}'
        }]
    )
    return response.content[0].text.strip()


# Run extraction for every PDF
car_sections = []   # list of (filename, markdown_section)

for pdf_path in pdf_files:
    print(f'\n📄 Processing: {pdf_path.name}')
    raw  = extract_text_from_pdf(str(pdf_path))
    print(f'   Extracted {len(raw)} chars → sending to Claude...')
    md   = extract_features_with_claude(raw, pdf_path.name)
    car_sections.append((pdf_path.name, md))
    print(f'   ✅ Got {len(md)} chars of Markdown')

print(f'\n✅ Done — {len(car_sections)} car(s) processed')

## Step 3 — Write combined Markdown file

In [ ]:
OUTPUT_MD = 'car_brochures.md'

lines = ['# Car Brochure Database\n']
for filename, section in car_sections:
    lines.append(section)
    lines.append('\n---\n')   # horizontal rule between cars

full_markdown = '\n'.join(lines)

with open(OUTPUT_MD, 'w', encoding='utf-8') as f:
    f.write(full_markdown)

print(f'✅ Written to {OUTPUT_MD} ({len(full_markdown)} chars)')
print('\n--- Preview (first 800 chars) ---')
print(full_markdown[:800])

## Step 4 — Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open(OUTPUT_MD, 'r', encoding='utf-8') as f:
    md_text = f.read()

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 400,
    chunk_overlap = 80,
    # Split on car sections first, then subsections, then paragraphs
    separators    = ['\n## ', '\n### ', '\n---', '\n\n', '\n', '. ']
)

chunks = splitter.split_text(md_text)

print(f'Total chunks: {len(chunks)}')
print(f'\nFirst chunk:\n{chunks[0]}')
print(f'\nSecond chunk:\n{chunks[1]}')

## Step 5 — Embedding

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

print('Loading embedding model...')
model = SentenceTransformer('intfloat/multilingual-e5-base')
DIM   = model.get_sentence_embedding_dimension()
print(f'✅ Model ready — dim: {DIM}')


def embed_passages(texts):
    return model.encode(
        [f'passage: {t}' for t in texts],
        normalize_embeddings=True,
        show_progress_bar=True
    )

def embed_query(query):
    return model.encode(f'query: {query}', normalize_embeddings=True)


print('\nEmbedding chunks...')
embeddings = embed_passages(chunks)
ids        = [str(i) for i in range(len(chunks))]

print(f'✅ Embeddings shape: {embeddings.shape}')

## Step 6 — ChromaDB index

In [ ]:
import chromadb

chroma = chromadb.PersistentClient(path='./chroma_cars')

# Fresh collection each run
try:
    chroma.delete_collection('car_brochures')
except Exception:
    pass

collection = chroma.create_collection(
    name     = 'car_brochures',
    metadata = {'hnsw:space': 'cosine'}
)

collection.add(
    ids        = ids,
    documents  = chunks,
    embeddings = embeddings.tolist()
    # No metadata — as requested
)

print(f'✅ ChromaDB ready — {collection.count()} chunks indexed')

## Step 7 — BM25 index

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r'\w+', text.lower())

bm25 = BM25Okapi([tokenize(c) for c in chunks])
print(f'✅ BM25 ready — {len(chunks)} docs')

## Step 8 — Hybrid Search

In [ ]:
def hybrid_search(query: str, k: int = 5, alpha: float = 0.7) -> list[dict]:
    """
    Hybrid search: vector (ChromaDB) + keyword (BM25).

    alpha = weight for vector score  (1-alpha = BM25 weight)
    Higher alpha → more semantic, lower → more keyword-exact.
    0.7 is good for sparse brochure text.
    """
    # ── Vector search ────────────────────────────────────────────────────────
    q_emb   = embed_query(query)
    results = collection.query(
        query_embeddings = [q_emb.tolist()],
        n_results        = min(20, len(chunks))
    )
    vector_scores = {
        int(vid): 1 / (1 + d)
        for vid, d in zip(results['ids'][0], results['distances'][0])
    }

    # ── BM25 keyword search ──────────────────────────────────────────────────
    bm25_raw = bm25.get_scores(tokenize(query))
    bm25_max = np.max(bm25_raw)
    bm25_norm = bm25_raw / bm25_max if bm25_max > 0 else bm25_raw

    # ── Combine scores ───────────────────────────────────────────────────────
    combined = {}
    for cid, vs in vector_scores.items():
        combined[cid] = alpha * vs
    for cid, bs in enumerate(bm25_norm):
        combined[cid] = combined.get(cid, 0) + (1 - alpha) * bs

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)

    return [
        {'chunk_id': cid, 'score': round(score, 4), 'text': chunks[cid]}
        for cid, score in ranked[:k]
    ]


def show(query, k=3):
    print(f'\n🔍 Query: {query!r}')
    print('=' * 60)
    for i, r in enumerate(hybrid_search(query, k=k), 1):
        print(f'\n#{i} | score={r["score"]} | chunk {r["chunk_id"]}')
        print(r['text'][:400])
        print('-' * 40)

print('✅ hybrid_search() and show() ready')

## Step 9 — Run Queries

In [ ]:
show('kapasitas mesin dan tenaga maksimum L300')
show('fitur keamanan Delica')
show('varian dan pilihan warna yang tersedia')
show('dimensi panjang dan lebar mobil')
show('sistem transmisi dan suspensi')

In [ ]:
# Try your own queries here
show('YOUR QUERY HERE', k=5)